# Loss-landscape lab

Interactive landscapes of the trained models: pick the model, the two directions, their
scales, the radius, the loss (own training loss, merit, merit after FS, or any weighted
combination of components), the data split, and the plot normalization.

Grids are cached in `lab`, so changing the loss, rho, normalization or view is instant;
only a new model / direction / radius / grid size recomputes.

**Speed.** Every grid is saved to `figures/landscape/lab_cache/` the first time it is computed,
so repeat views and kernel restarts are instant. The default views (N_EVAL=500, 31x31, random and
eigenvector planes at radius 0.1 / 0.5, the M1-M3-M4 plane) are precomputed on an H200 by
`precompute_lab_cache.py` (job `labCache`); with the defaults below they load instead of computing.

New grids: run the notebook on a GPU node, not the login node (login slices have 4 CPUs and are shared).
Easiest: ORCD OnDemand (https://orcd-ood.mit.edu) -> Jupyter, partition `pi_donti_gpu`, GPU `h200:1`.
Or: `srun -p pi_donti_gpu -G h200:1 --cpus-per-task=8 --mem=32G -t 2:00:00 --pty bash`, then
`jupyter lab --no-browser --ip=$(hostname) --port=8888` and tunnel the port.
On CPU (threads are capped at 4, override with `LAB_THREADS`), a 31x31 grid without FS is
~40 s per model; use `N = 15` and `N_EVAL = 200` for quick looks. `fs=True` is ~20-50x slower.

Helpers live in `landscape_lab.py`.

In [1]:
%matplotlib inline
import os
# must run before numpy/torch are imported (restart the kernel if they already are)
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, os.environ.get("LAB_THREADS", "4"))
import numpy as np
import matplotlib.pyplot as plt
from landscape_lab import Lab, show, load_npz

SPLIT = "test"      # "test" (held-out) or "train"
N_EVAL = 500        # samples per grid point
lab = Lab(split=SPLIT, n_eval=N_EVAL)

# Fig. 1 models (M1 SL small rho, M2 SL large rho, M3 SL+FS small rho, M4 SSL small rho; L1 penalty)
lab.add_from_json("figures/landscape/fig1/ckpts.json")
# any other checkpoint: lab.add("name", "results/.../model.pt")
for k in lab.models:
    print(lab.describe(k), " own loss =", lab.own_weights(k))

M1: sup_pen, rho=10.0, pen=l1, lr=0.0003  own loss = {'huber': 100.0, 'viol_l1': 10.0}
M2: sup_pen, rho=100000.0, pen=l1, lr=0.0003  own loss = {'huber': 100.0, 'viol_l1': 100000.0}
M3: sup_pen_fs, rho=10.0, pen=l1, lr=0.0003  own loss = {'huber_fs': 100.0, 'viol_l1': 10.0}
M4: penalty, rho=10.0, pen=l1, lr=0.0003  own loss = {'obj': 1.0, 'viol_l1': 10.0}


## 1. One model, your choice of directions

Direction specs: `("random", seed)`, `("eig", "figures/landscape/fig1/topvec_test_M1.pt")`
(top Hessian eigenvector, from `run_fig1_eig.sh`), or `("toward", "M4")` (straight at another model;
with `radius=1` the edge of the grid is that model). `DX_SCALE` / `DY_SCALE` stretch one axis.

Loss: `"own"`, `"merit"`, `"merit_fs"` (needs `FS=True`), or a dict such as
`{"huber": 100, "viol_l1": 1e3}` to try a different rho on the same grid.
Normalization: `"minmax"` (shape only), `"rise"` ((L - L0)/|L0|), `"none"`.

In [ ]:
MODEL = "M1"
DX, DY = ("random", 5), ("random", 6)
DX_SCALE, DY_SCALE = 1.0, 1.0
RADIUS, N = 0.5, 31
FS = False
LOSS = "own"
NORM, LOG = "minmax", True
STYLE = ("3d", "contour", "slice")

xs, ys, Z = lab.grid(MODEL, DX, DY, radius=RADIUS, n=N, dx_scale=DX_SCALE, dy_scale=DY_SCALE, fs=FS)
L = lab.loss(Z, LOSS, MODEL)
_ = show([(f"{MODEL}: {LOSS}", xs, ys, L)], style=STYLE, norm=NORM, log=LOG)

M1: row 2/31

## 2. All models side by side, same direction recipe

Each model gets its own directions from the same recipe (same seeds, filter-normalized to its
own weights). `SHARED_Z=True` puts every panel on one z scale.

In [ ]:
MODELS = ["M1", "M2", "M3", "M4"]
DX, DY = ("random", 5), ("random", 6)
# eigenvector view per model: DX = lambda m: ("eig", f"figures/landscape/fig1/topvec_test_{m}.pt")
RADIUS, N = 0.5, 31
FS = False
LOSS = "own"                      # or "merit", "merit_fs" (FS=True), or a dict
NORM, LOG, SHARED_Z = "minmax", True, True

panels = []
for m in MODELS:
    dx = DX(m) if callable(DX) else DX
    dy = DY(m) if callable(DY) else DY
    xs, ys, Z = lab.grid(m, dx, dy, radius=RADIUS, n=N, fs=FS)
    panels.append((lab.describe(m).split(",")[0] + f" | {LOSS}", xs, ys, lab.loss(Z, LOSS, m)))
_ = show(panels, style=("3d", "contour"), norm=NORM, log=LOG, shared_z=SHARED_Z)

## 3. Shared plane through three models (objective mismatch)

Every loss on the same xy; red dots are the models, white X is each loss's minimum.
`CLIP_PCT` clips rare FS-failure spikes so the rest stays readable.

In [ ]:
THREE = ["M1", "M3", "M4"]
N, MARGIN = 31, 0.5
FS = True                               # needed for merit_fs
LOSSES = [("own", "M1"), ("own", "M3"), ("own", "M4"), ("merit", None), ("merit_fs", None)]
NORM, LOG, CLIP_PCT = "none", True, 97

xs, ys, Z, pts = lab.plane(THREE, n=N, margin=MARGIN, fs=FS)
panels = [(f"{spec}" + (f" ({who})" if who else ""), xs, ys, lab.loss(Z, spec, who)) for spec, who in LOSSES]
_ = show(panels, style=("contour", "3d"), norm=NORM, log=LOG, shared_z=False, clip_pct=CLIP_PCT,
         points=[(p[0], p[1], n) for p, n in zip(pts, THREE)], azim=235, elev=45)

## 4. Browse precomputed grids (no compute)

Outputs of `compute_landscape_compare.py` / `run_fig1*.sh` (41x41, 1000 samples).
`prefix` is the model name inside the file (`"plane"` for plane files).

In [ ]:
import glob
print("\n".join(sorted(glob.glob("figures/landscape/fig1/*.npz"))))

In [ ]:
PATH, PREFIX, NAME = "figures/landscape/fig1/random_test_r0.5_M2.npz", "M2", "M2"
LOSS = "own"
xs, ys, L, pts = load_npz(PATH, PREFIX, lab, LOSS, NAME)
_ = show([(f"{PREFIX}: {LOSS}", xs, ys, L)], style=("3d", "contour", "slice"), norm="minmax")

## 5. Optional sliders

Needs `pip install ipywidgets` in the `ml4opt` env. Moving a slider re-plots from the cache;
a new radius / grid size / seed computes one new grid.

In [ ]:
try:
    import ipywidgets as W
except ImportError:
    W = None
    print("ipywidgets not installed: `pip install ipywidgets`, then re-run this cell")

if W is not None:
    @W.interact(model=list(lab.models), seed_x=W.IntSlider(5, 0, 50), seed_y=W.IntSlider(6, 0, 50),
                radius=W.FloatLogSlider(0.5, base=10, min=-2, max=0.5), n=W.IntSlider(21, 5, 61, 2),
                dx_scale=W.FloatSlider(1.0, min=0.1, max=5.0), loss=["own", "merit"],
                norm=["minmax", "rise", "none"], log=True, elev=W.IntSlider(35, 0, 90), azim=W.IntSlider(190, 0, 360))
    def _ui(model, seed_x, seed_y, radius, n, dx_scale, loss, norm, log, elev, azim):
        xs, ys, Z = lab.grid(model, ("random", seed_x), ("random", seed_y), radius=radius, n=n,
                             dx_scale=dx_scale, verbose=False)
        show([(f"{model}: {loss}", xs, ys, lab.loss(Z, loss, model))], style=("3d", "contour"),
             norm=norm, log=log, elev=elev, azim=azim)